# Milestone 5 Lab, notebook 1 of 7: Files arrive, by hand

## Where we are

| Notebook | What you do |
|---|---|
| **1. Files arrive, by hand** (you are here) | Load the files the naive way, and watch three things go wrong |
| 2. A loader that is safe to repeat | Fingerprint each file, check its columns, and choose a rule for every file |
| 3. AI reads a changed file | Ask a model what a renamed column means, and score its suggestions |
| 4. A workflow that can wait | Build the loader as a graph that pauses for your approval |
| 5. Crash and carry on | Restart the kernel on purpose, and find the file still waiting |
| 6. A swarm of file workers | Load fifty files side by side, and score the run for quality, cost, and time |
| 7. The same question, from loaded files | Approve the waiting files, run the pipeline, and ask the leader's question |

Model requests in this notebook: **0**. A full run of this lab costs about 3 US cents.

## What this notebook is for

In milestones 1 to 4 the business's records were seven complete tables that were simply there. A real business does not get its records that way. Its systems send **files**: this week's orders, this week's shipments, this month's costs.

In this notebook you load those files the simplest way anyone would, with no AI. It goes wrong in three different ways. The rest of the milestone fixes each one.

> If you have not done milestones 1 to 4, do them first.

In [ ]:
import pandas as pd
from lab_helpers import LANDING, load_result, show_doc

## Step 1: What the business is asking for now

In [ ]:
show_doc("BRD.md", "1. Business problem")

## Step 2: The landing folder

Files arrive in `data/landing/`. A **landing area** is where files wait before anything has looked at them. Nothing in it is trusted yet.

In [ ]:
files = sorted(file.name for file in LANDING.glob("*.csv"))

print(len(files), "files")
files[:12]

Each kind of file that keeps arriving is called a **feed**. The start of a file's name says which feed it belongs to, and the date says which week it covers.

In [ ]:
feeds = ["order_items", "orders", "payment_fees", "product_costs", "products", "refunds", "shipments"]

for feed in feeds:
    of_this_feed = [name for name in files if name.startswith(feed + "_") or name == feed + ".csv"]
    print(f"{feed:14} {len(of_this_feed)} files")

Open one. It is an ordinary CSV file with a line of column names and one line per record.

In [ ]:
print((LANDING / "orders_2026-09-14.csv").read_text(encoding="utf-8")[:230])

## Step 3: The naive loader

The simplest loader reads every file of a feed and puts them end to end. Three lines.

In [ ]:
def naive_load(feed):
    """Read every file of a feed and put them end to end."""
    feed_files = sorted(LANDING.glob(f"{feed}_*.csv"))
    tables = [pd.read_csv(file) for file in feed_files]
    return pd.concat(tables, ignore_index=True)


orders = naive_load("orders")

print(len(orders), "orders loaded")

## Step 4: First problem. A file arrived twice

In milestone 4 the business had 2,450 orders. The naive loader found more. Count how often each order id appears.

In [ ]:
times_seen = orders["order_id"].value_counts()
seen_twice = times_seen[times_seen > 1]

print("Expected 2,450 orders. Loaded", len(orders))
print("Orders that appear twice:", len(seen_twice))

Look at the file names again. One of them was sent twice.

In [ ]:
[name for name in files if name.startswith("orders_2026-09-14")]

In [ ]:
first = (LANDING / "orders_2026-09-14.csv").read_bytes()
second = (LANDING / "orders_2026-09-14_resend.csv").read_bytes()

print("The two files have the same content:", first == second)

The platform resent a file under a new name, and the naive loader loaded it again. A whole week of orders is now counted twice.

A name cannot protect us, because the name changed. Only the **content** is the same. Keep that in mind for notebook 2.

## Step 5: Second problem. A column was renamed

Now the order lines. Loading them raises no error.

In [ ]:
lines = naive_load("order_items")

print(len(lines), "order lines loaded")
lines.columns.tolist()

Eight columns, where we expect six. Two are new: `list_price` and `discount`. Look at how many values are missing in each column.

In [ ]:
lines.isna().sum()

From the week of 12 October, the platform started calling `unit_list_price` by a new name, `list_price`, and `discount_amount` became `discount`. pandas did not complain. It made new columns and left the old ones **empty** for those rows.

This is worse than a crash. Work out gross sales the way milestone 2 does, and compare with the figure you traced by hand in milestone 4.

In [ ]:
lineage_key = load_result("answer_key_lineage")

orders_once = orders.drop_duplicates()
bottle = lines[lines["product_id"] == "P001"].merge(orders_once, on="order_id")
bottle = bottle[bottle["order_date"].str.startswith("2026-10") & (bottle["status"] != "cancelled")]
bottle = bottle.drop_duplicates()

naive_gross_sales = (bottle["quantity"] * bottle["unit_list_price"]).sum()

print("Gross sales from the naive load:", naive_gross_sales)
print("The right figure:               ", lineage_key["gross_sales"])

More than two thirds of October's sales are gone, and nothing reported an error. A leader would simply see a bad month.

## Step 6: Third problem. A column is missing

One refund file arrived without its `return_handling_cost` column.

In [ ]:
incomplete = pd.read_csv(LANDING / "refunds_2026-10-12.csv")
complete = pd.read_csv(LANDING / "refunds_2026-10-12_complete.csv")

print("The first file's columns: ", incomplete.columns.tolist())
print("The second file's columns:", complete.columns.tolist())

The sender noticed and sent a complete file afterwards. The naive loader loads both. So that week's two refunds are counted twice, and one copy of each has no handling cost.

In [ ]:
refunds = naive_load("refunds")

print("Expected 50 refunds. Loaded", len(refunds))
print("Refunds with no handling cost:", refunds["return_handling_cost"].isna().sum())

## Step 7: Two more surprises that did no harm

Not every change is a problem. Two files changed in ways that should simply be accepted.

In [ ]:
print(pd.read_csv(LANDING / "refunds_2026-09-21.csv").columns.tolist())
print(pd.read_csv(LANDING / "shipments_2026-10-05.csv").columns.tolist())

The first has its columns in a different order. The second has an extra column, `warehouse_code`, that we do not use. A good loader should load both without fuss. It must tell these harmless changes apart from the harmful ones.

## What we now know

The naive loader failed in three ways, and none of them raised an error:

| What happened | What the naive loader did | What should happen |
|---|---|---|
| A file was sent twice | Loaded it twice: 282 orders doubled | Skip the second copy |
| Columns were renamed | Left the real columns empty: sales vanished | Notice, and find out what the new names mean |
| A column was missing | Loaded the file anyway | Reject the file and wait for a complete one |

- A file must be recognised by its **content**, because its name can change.
- A loader must compare each file's columns with what it **expects**, before loading anything.
- "No error" does not mean "loaded correctly".

**Next:** notebook 2, where you build a loader that is safe to repeat and has a rule for every kind of file.